# Azure Content Understanding — a hands-on companion notebook

Companion to the deck **Azure Content Understanding — Foundry Tools technical deep dive**.
The deck explains *what the service is and how it is built*. This notebook is the same story
in code: you build a small supplier-document pipeline and add one capability at a time.

**A note on naming.** Microsoft Learn identifies releases by **API version**, not by a product
version number — the labels "1.0" / "2.0" appear only in the Foundry blog. So this notebook is
explicit everywhere:

| | API version | Status |
|---|---|---|
| **GA** | `2025-11-01` | Generally available — use in production |
| **Preview** | `2026-06-01-preview` | No SLA — evaluate only |

Everything up to and including Step 6 runs on the **GA** API. Steps 7 and 8 are **preview only**
and are clearly marked.

**Everything here is native SDK / REST.** No helper classes, no wrappers, no framework of my own —
just `azure-ai-contentunderstanding` calls exactly as the service exposes them, so that anything
you learn transfers straight into your own code.

---

## The real-world problem we solve

You are in **accounts payable**. Suppliers send you a daily mix of PDFs — invoices, delivery notes,
statements, the occasional scanned bundle with several documents in one file. Today a person opens
each one and retypes the numbers.

We will build that up step by step:

| Step | What you add | Real-world question it answers |
|---|---|---|
| 0 | Connect + check resource defaults | Is my Foundry resource ready? |
| 1 | Read a document → Markdown | Can I make supplier PDFs searchable? |
| 2 | Local files, page ranges, LLM-ready text | Can I feed these into RAG without a parser? |
| 3 | Prebuilt `prebuilt-invoice` | Can I capture invoice fields with zero configuration? |
| 4 | Custom analyzer with your own schema | Can I get *my* fields off *my* document type? |
| 5 | Confidence + grounding → straight-through processing | Which ones can skip human review? |
| 6 | Classifier + segmentation | Can I split a mixed scanned batch automatically? |
| 7 | **Preview** — agentic workflow | Can it work out a value that is not written anywhere? |
| 8 | **Preview** — inline synchronous read | Can I get an answer without polling or storage? |

Steps 1–3 need no configuration at all. Steps 4–6 are where a production design actually lands.

## Before you start

You need four things. They are all one-time setup per Foundry resource.

1. **A Microsoft Foundry resource** in a region that supports Content Understanding.
   (The deck notes 12 regions at the time of writing, and that Microsoft's own pages are
   inconsistent on the count — confirm against the region-support page before you design.)
2. **The `Cognitive Services User` role** on that resource, assigned to you or to your service
   principal. This is required *even if you own the resource* — without it the calls that
   configure model deployments will fail.
3. **Model deployments** in that resource. Content Understanding brings no models of its own:
   the analyzer *names* a model and the resource maps that name to **your** deployment, which is
   why the token cost lands on your own Foundry bill and why provisioned throughput works.
   Deploy at least a completion model and `text-embedding-3-large`.
4. **Resource defaults configured**, mapping model names to your deployment names. You can do
   this in Content Understanding Studio (Settings → add resource), or programmatically —
   Step 0 below shows how to read them and how to set them.

### Environment variables

Copy `.env.example` to `.env` in the repository root and fill in the Content Understanding section.

| Variable | Required | Purpose |
|---|---|---|
| `CONTENTUNDERSTANDING_ENDPOINT` | **yes** | `https://<your-resource>.services.ai.azure.com/` |
| `CONTENTUNDERSTANDING_KEY` | no | API key. **Omit it** and the notebook uses `DefaultAzureCredential` (run `az login` first) — that is the recommended path. |
| `CU_COMPLETION_MODEL` | no | Model *name* the analyzer asks for. Default `gpt-5.2`, the documented recommendation. |
| `CU_EMBEDDING_MODEL` | no | Default `text-embedding-3-large`. |
| `CU_SAMPLE_DOC_PATH` | no | A local PDF for the file-based steps. The next section downloads one for you. |
| `CU_MIXED_BATCH_PATH` | no | A local PDF containing several documents, for Step 6. Also downloaded below. |
| `CU_AGENTIC_DOC_PATH` | no | A many-line-item invoice for Step 7. Also downloaded below. |

### Install

```bash
pip install -r requirements.txt
az login          # only if you are not using CONTENTUNDERSTANDING_KEY
```

In [ ]:
# Optional: install from inside the notebook.
# %pip install -r requirements.txt

---
## Sample documents

You do not need to supply your own PDFs to work through this notebook. Microsoft publishes
representative test documents in two public repositories, both MIT-licensed, and the cell below
downloads the ones this notebook uses.

**`Azure/azure-sdk-for-python`** → `sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_files/`
— the files the official SDK samples run against. These are the ones we use, because each was
purpose-built for a specific capability:

| File | Why it is the right test document |
|---|---|
| `sample_invoice.pdf` | A plain single invoice. The baseline for Steps 2, 4, 5 and 8. |
| `mixed_financial_docs.pdf` | **Several different document types in one file** — exactly the mailroom-batch problem Step 6 solves with classification and segmentation. |
| `workflow_invoice_20_items.pdf` | An invoice with **20 line items**, built to show the difference between standard and agentic extraction: a derived average over 20 rows is work the default workflow cannot do. Step 7. |
| `mixed_financial_invoices.pdf` | Multi-page, for experimenting with `content_range` page selection in Step 2. |

Worth knowing about, though this notebook does not use them:

| File | Capability it demonstrates |
|---|---|
| `sample_document_features.pdf` | Charts, hyperlinks, formulas and annotations |
| `sample_signature.png` | Signature detection (preview) |
| `sample_metadata.pdf` / `.docx` | Document metadata extraction (preview) |
| `mixed_financial_docs_in_page.pdf` | Splitting a **single page** into separately classified segments (preview) |
| `training_samples/` | Labelled data for the analyzer-improvement workflow |

**`Azure-Samples/azure-ai-content-understanding-python`** → `data/` — the solution-accelerator
repository, and the place to go for the **non-document modalities** this notebook only mentions:

| File | Modality |
|---|---|
| `invoice.pdf`, `mixed_financial_docs.pdf` | Documents |
| `receipt.png`, `pieChart.jpg` | Images |
| `audio.wav`, `callCenterRecording.mp3` | Audio — try these with `prebuilt-audioSearch` |
| `FlightSimulator.mp4` | Video — try this with `prebuilt-videoSearch` |

**If the download cell is blocked** by a corporate proxy, fetch the files by hand instead — open
the repository folder in a browser and download them into a `sample_files/` directory next to this
notebook, or clone with `git clone --depth 1 https://github.com/Azure/azure-sdk-for-python`. The
paths in `.env` are all that matter; nothing else depends on how the files got there.

> **For your own evaluation, do not stop at these.** Sample files are clean and cooperative. The
> confidence threshold in Step 5 is only meaningful when calibrated against *your* suppliers' real
> layouts — including the bad scans. Use these to learn the API; use your own documents to make
> the design decision.

In [1]:
import os
import urllib.request
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # .env wins over anything we set below

SAMPLE_DIR = Path("sample_files")
SAMPLE_DIR.mkdir(exist_ok=True)

SDK_SAMPLES_BASE = (
    "https://github.com/Azure/azure-sdk-for-python/raw/main"
    "/sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_files"
)

WANTED = [
    "sample_invoice.pdf",            # baseline single invoice
    "mixed_financial_docs.pdf",      # several document types in one file
    "workflow_invoice_20_items.pdf",  # 20 line items, for the agentic comparison
    "mixed_financial_invoices.pdf",  # multi-page, for content_range
]

for name in WANTED:
    target = SAMPLE_DIR / name
    if target.exists():
        print(f"already have  {name}")
        continue
    urllib.request.urlretrieve(f"{SDK_SAMPLES_BASE}/{name}", target)
    print(f"downloaded    {name}  ({target.stat().st_size:,} bytes)")

# Wire them into the environment variables the later steps read, unless your .env
# already points somewhere else.
os.environ.setdefault("CU_SAMPLE_DOC_PATH", str(SAMPLE_DIR / "sample_invoice.pdf"))
os.environ.setdefault("CU_MIXED_BATCH_PATH", str(SAMPLE_DIR / "mixed_financial_docs.pdf"))
os.environ.setdefault("CU_AGENTIC_DOC_PATH", str(SAMPLE_DIR / "workflow_invoice_20_items.pdf"))

print()
for var in ("CU_SAMPLE_DOC_PATH", "CU_MIXED_BATCH_PATH", "CU_AGENTIC_DOC_PATH"):
    print(f"{var:<22}= {os.environ[var]}")

downloaded    sample_invoice.pdf  (151,363 bytes)
downloaded    mixed_financial_docs.pdf  (266,116 bytes)
downloaded    workflow_invoice_20_items.pdf  (21,825 bytes)
downloaded    mixed_financial_invoices.pdf  (15,324 bytes)

CU_SAMPLE_DOC_PATH    = sample_files/sample_invoice.pdf
CU_MIXED_BATCH_PATH   = sample_files/mixed_financial_docs.pdf
CU_AGENTIC_DOC_PATH   = sample_files/workflow_invoice_20_items.pdf


---
## Step 0 — Connect, and check the resource is ready

Two objects and you are connected: a **credential** and a **client**.

`ContentUnderstandingClient` defaults to API version `2025-11-01` (GA). We create a second client
pinned to `2026-06-01-preview` further down, only for the two preview steps — keeping them
separate is deliberate, because mixing GA and preview is exactly how teams end up committing in
production to something that has no SLA.

In [ ]:
import os
import time
from typing import cast

from dotenv import load_dotenv

from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential

load_dotenv()

ENDPOINT = os.environ["CONTENTUNDERSTANDING_ENDPOINT"]
KEY = os.getenv("CONTENTUNDERSTANDING_KEY")

# Key auth is convenient for a test resource. Entra ID (DefaultAzureCredential) is the
# production path -- no secret in your .env, and role assignments do the authorising.
credential = AzureKeyCredential(KEY) if KEY else DefaultAzureCredential()

# GA client -- API version 2025-11-01 (the SDK default).
client = ContentUnderstandingClient(endpoint=ENDPOINT, credential=credential)

# Model NAMES the analyzers will ask for. The resource maps each name to one of YOUR deployments.
COMPLETION_MODEL = os.getenv("CU_COMPLETION_MODEL", "gpt-5.2")
EMBEDDING_MODEL = os.getenv("CU_EMBEDDING_MODEL", "text-embedding-3-large")

print(f"Endpoint : {ENDPOINT}")
print(f"Auth     : {'API key' if KEY else 'Entra ID (DefaultAzureCredential)'}")
print(f"Models   : completion={COMPLETION_MODEL}, embedding={EMBEDDING_MODEL}")

Endpoint : https://cog-tb7tpjtuee4ji.services.ai.azure.com/
Auth     : Entra ID (DefaultAzureCredential)
Models   : completion=gpt-5.2, embedding=text-embedding-3-large


### Are the resource defaults set?

`get_defaults()` returns the mapping from model *names* to your Foundry *deployments*. If this
comes back empty, prebuilt and custom analyzers will fail with a "model deployment not found"
error — fix it with `update_defaults()` before going further.

In [3]:
defaults = client.get_defaults()
print("Current model deployment defaults:")
print(defaults.as_dict())

Current model deployment defaults:
{'modelDeployments': {'gpt-4.1': 'gpt-4.1', 'gpt-4.1-mini': 'gpt-4.1-mini', 'text-embedding-3-large': 'text-embedding-3-large', 'prebuilt-analyzer-completion': 'gpt-5.5-831109', 'prebuilt-analyzer-completion-mini': 'gpt-5.2', 'prebuilt-analyzer-embedding': 'text-embedding-3-large', 'gpt-5.4-mini': 'gpt-5.4-mini', 'gpt-5-mini': 'gpt-5-mini', 'gpt-5.2': 'gpt-5.2', 'gpt-5.5': 'gpt-5.5-831109'}}


In [ ]:
# Run this ONCE per Foundry resource if the defaults above were empty.
# The keys are model names; the values are YOUR deployment names in Foundry.
#
# updated = client.update_defaults(
#     model_deployments={
#         COMPLETION_MODEL: os.environ["CU_COMPLETION_DEPLOYMENT"],
#         EMBEDDING_MODEL: os.environ["CU_EMBEDDING_DEPLOYMENT"],
#     }
# )
# print(updated.as_dict())

---
## Step 1 — Read a document into Markdown

**The problem:** a supplier PDF is a picture of information. Your search index and your language
model both want text.

**The simplest thing that works:** call a *prebuilt* analyzer. No configuration, no schema, no
training. `prebuilt-documentSearch` is the RAG-oriented one — it returns layout-preserving
Markdown plus a one-paragraph summary, which is exactly what a chunker and an embedding model
want to be fed.

Three things to notice in the code:

- **The file is referenced by URL.** The service fetches it. You can also send bytes — Step 2.
- **`begin_analyze` is a long-running operation.** It posts, gets `202 Accepted` with an
  operation location, and the poller handles the polling loop for you. `.result()` blocks until
  the status reads `Succeeded`.
- **`result.contents` is a list.** A single PDF is one `DocumentContent` item no matter how many
  pages it has; audio and video can come back as several segments.

In [4]:
from azure.ai.contentunderstanding.models import (
    AnalysisContentKind,
    AnalysisInput,
    AnalysisResult,
    DocumentContent,
)

# A sample invoice published by the Azure Content Understanding Python samples repository.
# Swap in any URL your Foundry resource can reach.
SAMPLE_INVOICE_URL = (
    "https://github.com/Azure-Samples/azure-ai-content-understanding-python"
    "/raw/refs/heads/main/data/invoice.pdf"
)

poller = client.begin_analyze(
    analyzer_id="prebuilt-documentSearch",
    inputs=[AnalysisInput(url=SAMPLE_INVOICE_URL)],
)
result: AnalysisResult = poller.result()

content = result.contents[0]
print(f"Content kind : {content.kind}")
print("\n--- Markdown (first 1500 characters) ---\n")
print(content.markdown[:1500])

Content kind : document

--- Markdown (first 1500 characters) ---

CONTOSO LTD.


# INVOICE

Contoso Headquarters
123 456th St
New York, NY, 10001

INVOICE: INV-100

INVOICE DATE: 11/15/2019

DUE DATE: 12/15/2019

CUSTOMER NAME: MICROSOFT CORPORATION

SERVICE PERIOD: 10/14/2019 - 11/14/2019

CUSTOMER ID: CID-12345

Microsoft Corp
123 Other St,
Redmond WA, 98052

BILL TO:
Microsoft Finance
123 Bill St,
Redmond WA, 98052

SHIP TO:
Microsoft Delivery
123 Ship St,
Redmond WA, 98052

SERVICE ADDRESS:
Microsoft Services
123 Service St,
Redmond WA, 98052


<table>
<tr>
<th>SALESPERSON</th>
<th>P.O. NUMBER</th>
<th>REQUISITIONER</th>
<th>SHIPPED VIA</th>
<th>F.O.B. POINT</th>
<th>TERMS</th>
</tr>
<tr>
<td></td>
<td>PO-3333</td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
</table>


<table>
<tr>
<th>DATE</th>
<th>ITEM CODE</th>
<th>DESCRIPTION</th>
<th>QTY</th>
<th>UM</th>
<th>PRICE</th>
<th>TAX</th>
<th>AMOUNT</th>
</tr>
<tr>
<td>3/4/2021</td>
<td>A123</td>
<td>Consulting Services</td>
<td>2

### What the structure looks like underneath

The Markdown is the readable view. `return_details` (on by default for the prebuilt search
analyzers) also gives you the structural detail — pages, tables, figures, paragraphs — which is
what you use when you want to chunk by section rather than by character count.

In [5]:
if content.kind == AnalysisContentKind.DOCUMENT:
    doc = cast(DocumentContent, content)
    print(f"MIME type  : {doc.mime_type or '(unknown)'}")
    print(f"Pages      : {doc.start_page_number} - {doc.end_page_number}")
    print(f"Page count : {len(doc.pages) if doc.pages else 0}")
    print(f"Tables     : {len(doc.tables) if doc.tables else 0}")

    for i, table in enumerate(doc.tables or [], start=1):
        print(f"  Table {i}: {table.row_count} rows x {table.column_count} columns")

MIME type  : application/pdf
Pages      : 1 - 1
Page count : 1
Tables     : 3
  Table 1: 2 rows x 6 columns
  Table 2: 4 rows x 8 columns
  Table 3: 5 rows x 2 columns


> **The other three content types work identically.** Swap the analyzer id and point at a
> different file: `prebuilt-imageSearch` for standalone images, `prebuilt-audioSearch` for call
> recordings (transcription with speaker diarisation), `prebuilt-videoSearch` for video (transcript
> plus key frames and shot detection). Same `begin_analyze` call, same result shape. That is the
> "one analyzer interface, four content types" claim from the deck, in practice.
>
> Try it with the multimodal samples from the solution-accelerator repository — pass the URL
> straight to `AnalysisInput(url=...)`, no download needed:
>
> ```python
> BASE = ("https://github.com/Azure-Samples/azure-ai-content-understanding-python"
>         "/raw/refs/heads/main/data")
>
> # Image
> client.begin_analyze("prebuilt-imageSearch",
>                      inputs=[AnalysisInput(url=f"{BASE}/pieChart.jpg")]).result()
> # Audio -- a call centre recording
> client.begin_analyze("prebuilt-audioSearch",
>                      inputs=[AnalysisInput(url=f"{BASE}/callCenterRecording.mp3")]).result()
> # Video
> client.begin_analyze("prebuilt-videoSearch",
>                      inputs=[AnalysisInput(url=f"{BASE}/FlightSimulator.mp4")]).result()
> ```
>
> One honest limit carried from the deck: **confidence scores and grounding are documents only** —
> not images, audio or video. A straight-through design over audio has no score to threshold on.

---
## Step 2 — Local files, page ranges, and LLM-ready output

**The problem:** your PDFs are not on a public URL. They are in a mailbox, a share, a blob
container you do not want to make reachable. And some are 200 pages when you only care about the
first three.

**What changes:** `begin_analyze_binary` sends the bytes in the request body instead of a URL.
`content_range` restricts the work — and therefore the bill, because content extraction is metered
per 1,000 pages.

`content_range` accepts `"1"`, `"1-3"`, `"9-"`, or a combination like `"1-3,5,9-"`. For documents
the numbers are 1-based page numbers; for audio and video they are milliseconds.

In [6]:
from azure.ai.contentunderstanding import to_llm_input

# Point this at a PDF of your own, or leave CU_SAMPLE_DOC_PATH unset to skip.
SAMPLE_DOC_PATH = os.getenv("CU_SAMPLE_DOC_PATH")

if SAMPLE_DOC_PATH and os.path.exists(SAMPLE_DOC_PATH):
    with open(SAMPLE_DOC_PATH, "rb") as f:
        file_bytes = f.read()

    # Analyze only the first three pages.
    local_poller = client.begin_analyze_binary(
        analyzer_id="prebuilt-documentSearch",
        binary_input=file_bytes,
        content_range="1-3",
    )
    local_result: AnalysisResult = local_poller.result()

    local_doc = cast(DocumentContent, local_result.contents[0])
    pages = [p.page_number for p in local_doc.pages] if local_doc.pages else []
    print(f"Analyzed pages: {pages}")
else:
    print("Set CU_SAMPLE_DOC_PATH to a local PDF to run this cell.")
    local_result = result  # fall back to the Step 1 result so the next cell still runs

Analyzed pages: [1]


### Handing the result to a language model

The Markdown can go straight into a prompt. The SDK also ships `to_llm_input()`, which packages a
result into one text block: YAML front matter (content type, page numbers, extracted fields)
followed by the Markdown body. It is a formatting convenience, not a wrapper — useful when you are
piping results into a vector store or an agent tool output and want a consistent shape.

In [7]:
llm_ready = to_llm_input(local_result)
print(llm_ready[:1200])

---
mimeType: application/pdf
metadata:
  contentType: application/pdf
  pageCount: '1'
  author: Anatoly Ponomarev
  createdAt: '2021-03-04T12:37:18-08:00'
  language: en-US
pages: 1
---
<!-- InputPageNumber: 1 -->

CONTOSO LTD.


# INVOICE

Contoso Headquarters
123 456th St
New York, NY, 10001

INVOICE: INV-100

INVOICE DATE: 11/15/2019

DUE DATE: 12/15/2019

CUSTOMER NAME: MICROSOFT CORPORATION

SERVICE PERIOD: 10/14/2019 - 11/14/2019

CUSTOMER ID: CID-12345

Microsoft Corp
123 Other St,
Redmond WA, 98052

BILL TO:
Microsoft Finance
123 Bill St,
Redmond WA, 98052

SHIP TO:
Microsoft Delivery
123 Ship St,
Redmond WA, 98052

SERVICE ADDRESS:
Microsoft Services
123 Service St,
Redmond WA, 98052


<table>
<tr>
<th>SALESPERSON</th>
<th>P.O. NUMBER</th>
<th>REQUISITIONER</th>
<th>SHIPPED VIA</th>
<th>F.O.B. POINT</th>
<th>TERMS</th>
</tr>
<tr>
<td></td>
<td>PO-3333</td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
</table>


<table>
<tr>
<th>DATE</th>
<th>ITEM CODE</th>
<th>DESCRIPTION</

---
## Step 3 — Typed fields with zero configuration

**The problem:** Markdown is still prose. Accounts payable needs `InvoiceTotal` as a number it can
post to the ledger, not a string it has to find in a paragraph.

**What changes:** switch from a *content* analyzer to a *domain* analyzer. `prebuilt-invoice` has a
field schema already defined, built on knowledge bases of real-world invoices, so it handles layout
variation between suppliers without you configuring anything.

This is the shift the deck describes: OCR returns whatever text is on the page and your code goes
hunting; a field schema means **you declare what you want and the service returns exactly that**,
as typed JSON.

Note the two shapes a field can take. A scalar field exposes `.value`. An **array** field — like
`Items` — exposes a list, and each element is an **object** field whose `.value_object` is itself a
dictionary of fields. That nesting is how line items come back.

In [8]:
invoice_poller = client.begin_analyze(
    analyzer_id="prebuilt-invoice",
    inputs=[AnalysisInput(url=SAMPLE_INVOICE_URL)],
)
invoice_result: AnalysisResult = invoice_poller.result()
invoice_doc = cast(DocumentContent, invoice_result.contents[0])

fields = invoice_doc.fields or {}
print(f"Fields returned: {len(fields)}\n")

for name in ("VendorName", "CustomerName", "InvoiceId", "InvoiceDate", "InvoiceTotal"):
    field = fields.get(name)
    print(f"{name:<15}: {field.value if field else '(not found)'}")

Fields returned: 31

VendorName     : CONTOSO LTD.
CustomerName   : MICROSOFT CORPORATION
InvoiceId      : INV-100
InvoiceDate    : 2019-11-15
InvoiceTotal   : (not found)


In [9]:
# Line items: an array field whose elements are object fields.
items = fields.get("Items")
if items and items.value:
    print(f"\n{len(items.value)} line item(s):")
    for i, item in enumerate(items.value, start=1):
        props = item.value_object or {}
        description = props.get("Description")
        quantity = props.get("Quantity")
        unit_price = props.get("UnitPrice")
        amount = props.get("Amount")
        print(
            f"  {i}. {description.value if description else '(none)'}"
            f" | qty {quantity.value if quantity else '-'}"
            f" | unit {unit_price.value if unit_price else '-'}"
            f" | amount {amount.value if amount else '-'}"
        )
else:
    print("\nNo line items returned for this document.")


No line items returned for this document.


---
## Step 4 — Your own fields, on your own document type

**The problem:** `prebuilt-invoice` is excellent at invoices. It knows nothing about your
supplier **delivery notes**, and it has never heard of your internal PO numbering.

**What changes:** you create a **custom analyzer**. This is the central object in the whole
service — a named, reusable configuration holding the extraction settings, your field schema, and
which models to use. You create it once and reuse it across every file of that type.

There is **no training step and no labelling**. A field *description in plain language is the
instruction*. That is the thing to internalise: you are not writing prompts per document type,
you are declaring fields.

### The three extraction methods

Choosing the method per field is most of the craft here:

| Method | What it does | Use it when |
|---|---|---|
| `EXTRACT` | Returns the value **as it appears** in the content | A literal on the page — a PO number, a date, a total. Documents only, and needs source/confidence enabled. |
| `GENERATE` | The model **interprets** and produces a value | Summaries, inferred values, anything not written verbatim. |
| `CLASSIFY` | Picks from a **fixed `enum`** you supply | Status, category, type — a closed set of answers. |

Omit `method` and the service picks based on the field type and description.

### Grounding and confidence

`estimate_field_source_and_confidence=True` on the config is what makes the service return, for
each value, **where it came from** and **how sure it is**. Per the deck, that computation happens in
the contextualization layer — and it is what Step 5 is built on. Turn it on at the analyzer level;
individual fields can override it.

In [10]:
from azure.ai.contentunderstanding.models import (
    ContentAnalyzer,
    ContentAnalyzerConfig,
    ContentFieldDefinition,
    ContentFieldSchema,
    ContentFieldType,
    GenerationMethod,
)

DELIVERY_NOTE_ANALYZER_ID = f"ap_delivery_note_{int(time.time())}"

field_schema = ContentFieldSchema(
    name="supplier_delivery_note",
    description="Fields accounts payable needs from a supplier delivery note",
    fields={
        # --- EXTRACT: literal values printed on the page ---
        "SupplierName": ContentFieldDefinition(
            type=ContentFieldType.STRING,
            method=GenerationMethod.EXTRACT,
            description="Legal name of the supplier issuing the delivery note",
            estimate_source_and_confidence=True,
        ),
        "PurchaseOrderNumber": ContentFieldDefinition(
            type=ContentFieldType.STRING,
            method=GenerationMethod.EXTRACT,
            description="Purchase order number the delivery relates to",
            estimate_source_and_confidence=True,
        ),
        "DeliveryDate": ContentFieldDefinition(
            type=ContentFieldType.DATE,
            method=GenerationMethod.EXTRACT,
            description="Date the goods were delivered",
            estimate_source_and_confidence=True,
        ),
        "TotalAmount": ContentFieldDefinition(
            type=ContentFieldType.NUMBER,
            method=GenerationMethod.EXTRACT,
            description="Total value of the goods delivered, excluding tax",
            estimate_source_and_confidence=True,
        ),
        # --- CLASSIFY: a closed set of answers ---
        "DeliveryStatus": ContentFieldDefinition(
            type=ContentFieldType.STRING,
            method=GenerationMethod.CLASSIFY,
            description=(
                "Whether the delivery was complete, partial, or the document records a "
                "return of goods"
            ),
            enum=["complete", "partial", "return", "unknown"],
        ),
        # --- GENERATE: not written anywhere verbatim ---
        "ExceptionNote": ContentFieldDefinition(
            type=ContentFieldType.STRING,
            method=GenerationMethod.GENERATE,
            description=(
                "One sentence describing any damage, shortage or discrepancy recorded on the "
                "note. Say 'none' if the delivery is clean."
            ),
        ),
    },
)

config = ContentAnalyzerConfig(
    return_details=True,
    enable_ocr=True,
    enable_layout=True,
    # This is what produces the confidence score and the source region per field.
    estimate_field_source_and_confidence=True,
    table_format="html",
)

analyzer = ContentAnalyzer(
    base_analyzer_id="prebuilt-document",      # prebuilt-image / -audio / -video for other modalities
    description="Accounts payable: supplier delivery notes",
    config=config,
    field_schema=field_schema,
    models={                                    # required whenever you supply a field_schema
        "completion": COMPLETION_MODEL,
        "embedding": EMBEDDING_MODEL,
    },
)

create_poller = client.begin_create_analyzer(
    analyzer_id=DELIVERY_NOTE_ANALYZER_ID,
    resource=analyzer,
    allow_replace=True,
)
created = create_poller.result()
print(f"Created analyzer: {DELIVERY_NOTE_ANALYZER_ID}")

Created analyzer: ap_delivery_note_1790766041


In [11]:
# Read it back -- an analyzer is a durable, inspectable resource, not a one-off request.
stored = client.get_analyzer(analyzer_id=DELIVERY_NOTE_ANALYZER_ID)
print(f"Description : {stored.description}")
print(f"Base        : {stored.base_analyzer_id}")
print(f"Status      : {stored.status}")
print("Fields:")
for field_name, definition in (stored.field_schema.fields or {}).items():
    print(f"  - {field_name:<22} {definition.type}  ({definition.method or 'auto'})")

Description : Accounts payable: supplier delivery notes
Base        : prebuilt-document
Status      : ContentAnalyzerStatus.READY
Fields:
  - SupplierName           ContentFieldType.STRING  (GenerationMethod.EXTRACT)
  - PurchaseOrderNumber    ContentFieldType.STRING  (GenerationMethod.EXTRACT)
  - DeliveryDate           ContentFieldType.DATE  (GenerationMethod.EXTRACT)
  - TotalAmount            ContentFieldType.NUMBER  (GenerationMethod.EXTRACT)
  - DeliveryStatus         ContentFieldType.STRING  (GenerationMethod.CLASSIFY)
  - ExceptionNote          ContentFieldType.STRING  (GenerationMethod.GENERATE)


Now run a document through it. The calling pattern is **identical** to the prebuilt analyzers —
that is the point of the analyzer abstraction. You changed the configuration, not the code path.

In [12]:
# Use your own delivery note if you have one; otherwise this demonstrates the call shape
# against the sample invoice (fields that do not apply simply come back empty).
DOC_UNDER_TEST = os.getenv("CU_SAMPLE_DOC_PATH")

if DOC_UNDER_TEST and os.path.exists(DOC_UNDER_TEST):
    with open(DOC_UNDER_TEST, "rb") as f:
        custom_poller = client.begin_analyze_binary(
            analyzer_id=DELIVERY_NOTE_ANALYZER_ID,
            binary_input=f.read(),
        )
else:
    custom_poller = client.begin_analyze(
        analyzer_id=DELIVERY_NOTE_ANALYZER_ID,
        inputs=[AnalysisInput(url=SAMPLE_INVOICE_URL)],
    )

custom_result: AnalysisResult = custom_poller.result()
custom_doc = cast(DocumentContent, custom_result.contents[0])

for field_name, field in (custom_doc.fields or {}).items():
    print(f"{field_name:<22}: {field.value}")

SupplierName          : CONTOSO LTD.
PurchaseOrderNumber   : PO-3333
DeliveryDate          : 2021-03-06
TotalAmount           : 100.0
DeliveryStatus        : unknown
ExceptionNote         : none


---
## Step 5 — Confidence and grounding: straight-through processing

This is the commercially interesting step, and the reason the whole exercise pays for itself.

**The problem:** an extracted value with no measure of trust leaves you with two bad options —
review everything by hand, or review nothing and absorb the errors.

**What changes:** because Step 4 enabled `estimate_field_source_and_confidence`, every value now
arrives with a **confidence score between 0 and 1** and a **source region** pointing at the place in
the document it came from. So you set a threshold: everything above it posts automatically,
everything below it goes to a person — *with the evidence attached*, so the reviewer checks a
highlighted region rather than re-reading the document.

Two cautions worth carrying into a design review:

- **Measure the threshold, do not guess it.** Run a representative sample of your own documents and
  look at where the errors actually fall. The number below is a placeholder.
- **Do not economise on the model here.** Microsoft's own guidance notes that confidence quality
  drops noticeably on the mini and nano model variants. If your threshold is load-bearing, use the
  full model.

In [13]:
CONFIDENCE_THRESHOLD = 0.85   # placeholder -- calibrate on your own documents

auto_approved = {}
needs_review = []

for field_name, field in (custom_doc.fields or {}).items():
    confidence = getattr(field, "confidence", None)
    source = getattr(field, "source", None)

    if confidence is not None and confidence >= CONFIDENCE_THRESHOLD:
        auto_approved[field_name] = field.value
    else:
        needs_review.append((field_name, field.value, confidence, source))

print(f"Threshold: {CONFIDENCE_THRESHOLD}\n")

print(f"Straight through ({len(auto_approved)}):")
for field_name, value in auto_approved.items():
    print(f"  {field_name:<22}: {value}")

print(f"\nRouted to a reviewer ({len(needs_review)}):")
for field_name, value, confidence, source in needs_review:
    shown = f"{confidence:.3f}" if confidence is not None else "no score"
    print(f"  {field_name:<22}: {value}   [confidence {shown}]")
    if source:
        # The source region is what the reviewer's UI highlights.
        print(f"      evidence: {source}")

Threshold: 0.85

Straight through (2):
  SupplierName          : CONTOSO LTD.
  PurchaseOrderNumber   : PO-3333

Routed to a reviewer (4):
  DeliveryDate          : 2021-03-06   [confidence 0.379]
      evidence: D(1,0.5753,6.2681,1.2150,6.2681,1.2150,6.4292,0.5753,6.4292)
  TotalAmount           : 100.0   [confidence 0.771]
      evidence: D(1,7.3650,6.8594,7.9272,6.8529,7.9291,7.0142,7.3669,7.0207)
  DeliveryStatus        : unknown   [confidence 0.298]
      evidence: D(1,7.0474,0.5704,8.0061,0.5686,8.0061,0.7926,7.0474,0.7920);D(1,6.8312,1.4015,8.0103,1.3989,8.0106,1.5464,6.8315,1.5490);D(1,6.2007,1.5901,8.0062,1.5909,8.0061,1.7522,6.2006,1.7515);D(1,6.4744,1.8085,8.0061,1.8063,8.0063,1.9642,6.4746,1.9664);D(1,0.7196,5.4002,1.0884,5.3985,1.0884,5.5376,0.7196,5.5376);D(1,0.5738,5.6652,1.2150,5.6728,1.2150,5.8339,0.5743,5.8264);D(1,0.5748,5.9758,1.2171,5.9787,1.2171,6.1398,0.5753,6.1369);D(1,0.5753,6.2681,1.2150,6.2681,1.2150,6.4292,0.5753,6.4292);D(1,6.0056,6.8911,6.7361,6.8911,6.736

> **The decision this drives.** If, on your documents, 80% of fields clear the threshold, you have
> removed 80% of the keying work and kept a human on the 20% that genuinely needs one. That ratio —
> measured, not assumed — is the business case. It is also why the deck insists confidence and
> grounding are documents-only: a straight-through design over images or audio has nothing to
> threshold on.

---
## Step 6 — A mixed scanned batch, split and routed automatically

**The problem:** the post room scans a stack. One PDF arrives holding an invoice, then a delivery
note, then a bank statement. Nobody knows what is in it until somebody opens it.

**What changes:** a **classifier** is just a custom analyzer configured with
`content_categories` — a map of category names to plain-language descriptions. Then:

- `enable_segment=False` → classify the **whole file** as one category.
- `enable_segment=True` → **split** the file into segments and classify each one.

Classification and content extraction happen in **one API call**, and each detected segment can be
routed onward to the analyzer best suited to it. You can define up to 200 categories, nested up to
5 levels. Include an `Other` category unless you want every file forced into one of your defined
buckets.

In [14]:
from azure.ai.contentunderstanding.models import ContentCategoryDefinition

CLASSIFIER_ID = f"ap_mailroom_triage_{int(time.time())}"

categories = {
    "Invoice": ContentCategoryDefinition(
        description=(
            "A request for payment issued by a supplier, listing goods or services, "
            "quantities, prices, tax and a total due, with payment terms."
        )
    ),
    "Delivery_Note": ContentCategoryDefinition(
        description=(
            "A document accompanying a shipment that lists the goods delivered and the "
            "purchase order they relate to. Usually has no prices or payment terms."
        )
    ),
    "Bank_Statement": ContentCategoryDefinition(
        description=(
            "A statement issued by a bank summarising account activity over a period: "
            "deposits, withdrawals, fees and running balances."
        )
    ),
    "Other": ContentCategoryDefinition(
        description="Anything that does not clearly belong to one of the other categories."
    ),
}

classifier = ContentAnalyzer(
    base_analyzer_id="prebuilt-document",
    description="Accounts payable mailroom triage",
    config=ContentAnalyzerConfig(
        return_details=True,
        content_categories=categories,
        enable_segment=True,     # split a multi-document file; False = classify the file as a whole
    ),
    models={"completion": COMPLETION_MODEL},
)

classifier_poller = client.begin_create_analyzer(
    analyzer_id=CLASSIFIER_ID,
    resource=classifier,
    allow_replace=True,
)
classifier_poller.result()
print(f"Created classifier: {CLASSIFIER_ID}")

Created classifier: ap_mailroom_triage_1790766142


In [15]:
# Point CU_MIXED_BATCH_PATH at a PDF containing several documents to see segmentation work.
MIXED_BATCH_PATH = os.getenv("CU_MIXED_BATCH_PATH")

if MIXED_BATCH_PATH and os.path.exists(MIXED_BATCH_PATH):
    with open(MIXED_BATCH_PATH, "rb") as f:
        batch_poller = client.begin_analyze_binary(
            analyzer_id=CLASSIFIER_ID,
            binary_input=f.read(),
        )
else:
    batch_poller = client.begin_analyze(
        analyzer_id=CLASSIFIER_ID,
        inputs=[AnalysisInput(url=SAMPLE_INVOICE_URL)],
    )

batch_result: AnalysisResult = batch_poller.result()
batch_doc = cast(DocumentContent, batch_result.contents[0])

print(f"File pages: {batch_doc.start_page_number}-{batch_doc.end_page_number}\n")

if batch_doc.segments:
    print(f"{len(batch_doc.segments)} segment(s) detected:")
    for segment in batch_doc.segments:
        print(f"  {segment.category or '(unknown)':<16} "
              f"pages {segment.start_page_number}-{segment.end_page_number}")
else:
    print("No segments -- the file was classified as a single document.")

File pages: 1-4

3 segment(s) detected:
  Invoice          pages 1-1
  Bank_Statement   pages 2-3
  Other            pages 4-4


> **Where this goes in a real pipeline.** Each segment's category becomes a routing decision:
> `Invoice` → `prebuilt-invoice`; `Delivery_Note` → the custom analyzer from Step 4;
> `Other` → a human. One call replaces a separate classification service plus your own splitting
> code.

---
## Step 7 — Preview: agentic workflow, for answers that are not written down

> ⚠️ **Preview — API `2026-06-01-preview`. No SLA. Evaluate only; do not commit production to it.**

**The problem:** standard extraction finds a value that is *written somewhere*. But "does this
invoice total match the purchase order?" is not printed on either document. Neither is "what is the
average unit price across these 40 lines?" Those answers have to be **constructed** from several
parts of the document.

**What changes:** exactly one setting — `workflow=ContentAnalyzerWorkflow.AGENTIC` on the analyzer
config. Everything else is the same: the same schema, the same call, the same typed result. The
reasoning is internal. It reads across separated parts, calculates derived values, validates
against conditions you wrote into the field descriptions, and interprets tables and figures as
evidence.

**Know the cost before you plan on it:**

| | |
|---|---|
| Billing | The higher *advanced contextualization* rate |
| Tokens | More than a standard workflow, on your own deployment |
| Latency | Typically longer for the same document |
| Capacity | About **400,000 tokens per minute per job** — a provisioning decision, not a detail |

**And the preview limits:** documents only · **one input file per request** · fields using the
`extract` method are **not supported** · labelled samples cannot improve it · Microsoft states
plainly that it is **not a replacement for human review**.

Because `extract` is unsupported here, the schema below uses `generate` throughout.

In [16]:
# A second client, pinned to the preview API version. Keeping it separate from `client`
# is deliberate -- GA and preview should never be ambiguous in your codebase.
preview_client = ContentUnderstandingClient(
    endpoint=ENDPOINT,
    credential=credential,
    api_version="2026-06-01-preview",
)

# ContentAnalyzerWorkflow ships with the SDK version that supports 2026-06-01-preview.
from azure.ai.contentunderstanding.models import ContentAnalyzerWorkflow

AGENTIC_ANALYZER_ID = f"ap_invoice_reconciliation_{int(time.time())}"

reconciliation_schema = ContentFieldSchema(
    name="invoice_reconciliation",
    description="Derived and validated values an AP clerk would otherwise work out by hand",
    fields={
        "LineItemCount": ContentFieldDefinition(
            type=ContentFieldType.INTEGER,
            method=GenerationMethod.GENERATE,
            description="How many distinct line items appear on the invoice",
        ),
        "AverageLinePrice": ContentFieldDefinition(
            type=ContentFieldType.NUMBER,
            method=GenerationMethod.GENERATE,
            description=(
                "The average amount per line item, calculated from the line item amounts "
                "on the invoice"
            ),
        ),
        "TotalsReconcile": ContentFieldDefinition(
            type=ContentFieldType.BOOLEAN,
            method=GenerationMethod.GENERATE,
            description=(
                "True if the sum of the line item amounts, plus any tax and shipping shown, "
                "equals the stated invoice total. False if they disagree."
            ),
        ),
        "ReconciliationNote": ContentFieldDefinition(
            type=ContentFieldType.STRING,
            method=GenerationMethod.GENERATE,
            description=(
                "If the totals do not reconcile, state the discrepancy and the amount. "
                "Otherwise say 'reconciled'."
            ),
        ),
    },
)

agentic_analyzer = ContentAnalyzer(
    base_analyzer_id="prebuilt-document",
    description="Invoice reconciliation using the agentic workflow (preview)",
    field_schema=reconciliation_schema,
    config=ContentAnalyzerConfig(
        return_details=True,
        workflow=ContentAnalyzerWorkflow.AGENTIC,   # <-- the entire difference
    ),
    models={"completion": COMPLETION_MODEL},
)

agentic_poller = preview_client.begin_create_analyzer(
    analyzer_id=AGENTIC_ANALYZER_ID,
    resource=agentic_analyzer,
    allow_replace=True,
)
agentic_poller.result()
print(f"Created agentic analyzer: {AGENTIC_ANALYZER_ID}")

Created agentic analyzer: ap_invoice_reconciliation_1790766218


In [17]:
# workflow_invoice_20_items.pdf is the right test here: 20 line items means the average
# and the reconciliation are genuinely derived values, not something printed on the page.
# One input file per request in this preview. Expect longer latency than a standard analyzer.
AGENTIC_DOC_PATH = os.getenv("CU_AGENTIC_DOC_PATH")

if AGENTIC_DOC_PATH and os.path.exists(AGENTIC_DOC_PATH):
    with open(AGENTIC_DOC_PATH, "rb") as f:
        agentic_result: AnalysisResult = preview_client.begin_analyze_binary(
            analyzer_id=AGENTIC_ANALYZER_ID,
            binary_input=f.read(),
        ).result()
else:
    agentic_result = preview_client.begin_analyze(
        analyzer_id=AGENTIC_ANALYZER_ID,
        inputs=[AnalysisInput(url=SAMPLE_INVOICE_URL)],
    ).result()

agentic_doc = cast(DocumentContent, agentic_result.contents[0])

for field_name, field in (agentic_doc.fields or {}).items():
    print(f"{field_name:<22}: {field.value}")

LineItemCount         : 20
AverageLinePrice      : 55.9575
TotalsReconcile       : True
ReconciliationNote    : reconciled


> **When *not* to use it.** For straightforward field extraction the standard analyzer is cheaper
> and faster. Reach for agentic mode only when the answer genuinely has to be reasoned out —
> reconciling figures, checking a document against a set of requirements, relating a contract to
> its amendments inside the same file.

---
## Step 8 — Preview: inline synchronous read, no polling and no storage

> ⚠️ **Preview — API `2026-06-01-preview`. No SLA.**

**The problem:** a user drags a document onto your web app and waits. Submit-and-poll is fine for a
nightly batch, awkward for an interactive screen. And sometimes a privacy review objects not to the
latency but to the service *storing* the content at all.

**What changes:** `analyze_binary_inline` (and `analyze_inline` for URLs) returns the answer **in
the response**. No poller, no operation location. The content is processed **in memory rather than
stored by the service** — and in practice that, not speed, is usually the deciding factor.

**The trade-offs are real, and it is not "a faster version of the service":**

| | Asynchronous (GA) | Inline (preview) |
|---|---|---|
| Content types | Documents, images, audio, video | **Documents only** |
| Analyzers | All | Read / layout / digital-parse, and custom document analyzers **without fields** |
| Field extraction | Yes — your schema, with confidence and source | **No custom fields** |
| Size ceiling | 200 MB, up to 300 pages | **10 MB, first 5 pages** |
| Cost | Standard page rate | **~50% more per page** |

**The rule that follows:** if you need your own fields, you are on the asynchronous path regardless.

In [18]:
from azure.ai.contentunderstanding.models import ContentAnalyzerInlineResponse

if SAMPLE_DOC_PATH and os.path.exists(SAMPLE_DOC_PATH):
    with open(SAMPLE_DOC_PATH, "rb") as f:
        inline_bytes = f.read()

    # No poller -- the result comes back in the response.
    inline_response: ContentAnalyzerInlineResponse = preview_client.analyze_binary_inline(
        analyzer_id="prebuilt-layout",   # read / layout / digitalParse only
        binary_input=inline_bytes,
        content_range="1-3",             # inline supports at most 5 pages
    )
    inline_result: AnalysisResult = inline_response.result
    inline_doc = cast(DocumentContent, inline_result.contents[0])

    print(f"Pages: {inline_doc.start_page_number}-{inline_doc.end_page_number}")
    print("\n--- Markdown (first 800 characters) ---\n")
    print(inline_doc.markdown[:800])
else:
    print("Set CU_SAMPLE_DOC_PATH to a local PDF (<=5 pages of interest) to run this cell.")

Pages: 1-1

--- Markdown (first 800 characters) ---

CONTOSO LTD.


# INVOICE

Contoso Headquarters
123 456th St
New York, NY, 10001

INVOICE: INV-100

INVOICE DATE: 11/15/2019

DUE DATE: 12/15/2019

CUSTOMER NAME: MICROSOFT CORPORATION

SERVICE PERIOD: 10/14/2019 - 11/14/2019

CUSTOMER ID: CID-12345

Microsoft Corp
123 Other St,
Redmond WA, 98052

BILL TO:
Microsoft Finance
123 Bill St,
Redmond WA, 98052

SHIP TO:
Microsoft Delivery
123 Ship St,
Redmond WA, 98052

SERVICE ADDRESS:
Microsoft Services
123 Service St,
Redmond WA, 98052


<table>
<tr>
<th>SALESPERSON</th>
<th>P.O. NUMBER</th>
<th>REQUISITIONER</th>
<th>SHIPPED VIA</th>
<th>F.O.B. POINT</th>
<th>TERMS</th>
</tr>
<tr>
<td></td>
<td>PO-3333</td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
</table>


<table>
<tr>
<th>DATE</th>
<th>ITEM CODE</th>
<th>DESCRIPTION</th>
<th>QTY</th>



---
## Step 9 — Housekeeping

Two things worth building into any pipeline from day one.

**Delete the result when you are done with it.** The asynchronous path stores the analysis result
so you can poll for it. Once you have retrieved it you can explicitly remove it — which matters
when the content is sensitive and your retention policy is stricter than the service default.

**Analyzers are durable resources.** They persist until you delete them, they consume your
namespace, and stale ones are a real source of confusion six months later. List and clean up.

In [ ]:
# Analysis results can be explicitly deleted once retrieved.
print(f"Operation id of the Step 3 analysis: {invoice_poller.operation_id}")
client.delete_result(operation_id=invoice_poller.operation_id)
print("Result deleted.")

In [ ]:
print("Analyzers on this resource:")
for a in client.list_analyzers():
    print(f"  {a.analyzer_id:<40} {a.description or ''}")

In [ ]:
# Clean up the analyzers this notebook created.
for analyzer_id, owning_client in (
    (DELIVERY_NOTE_ANALYZER_ID, client),
    (CLASSIFIER_ID, client),
    (AGENTIC_ANALYZER_ID, preview_client),
):
    try:
        owning_client.delete_analyzer(analyzer_id=analyzer_id)
        print(f"Deleted {analyzer_id}")
    except Exception as exc:          # already gone, or never created if a cell was skipped
        print(f"Skipped {analyzer_id}: {exc}")

---
## What constrains a design, and what it costs

Carried over from the deck, because these are the numbers that shape an architecture conversation.

**Limits**

| | |
|---|---|
| File size | 200 MB, up to 300 pages |
| Field schema | Up to 1,000 fields per analyzer |
| Classification | Up to 200 categories, nested up to 5 levels |
| Confidence and grounding | **Documents only** — not images, audio or video |
| Throughput | 1,000 pages or images a minute; 4 hours of audio or video |
| Regions | 12 at the time of writing — confirm before you design |
| Inline (preview) | 10 MB, first 5 pages |

**The bill has three parts, and only two appear on this service**

1. **Content extraction** — per 1,000 pages. The rate depends on the work *actually done*, not the
   analyzer you asked for: a digital document needing no OCR is charged at the lowest tier even if
   you called a layout analyzer.
2. **Contextualization** — per million tokens. The preview *advanced* tier bills at a higher rate.
3. **Your own model deployment** — language and embedding tokens land on **your** Foundry bill, not
   on this service's meter. This is the single most common surprise in a first invoice.

**One more thing the deck flags:** processing time for the asynchronous path is not published as a
guaranteed figure. Measure it on your own documents — and note that the Azure AI Search integration
times out at five minutes.

---

## Where to go next

- **Improve an analyzer with labelled examples** — when a new vendor's layout produces wrong values
  or low confidence, upload a sample, auto-label, correct, rebuild. One sample is enough when
  templates barely vary; one per format when they do. In preview, the patterns are *distilled* into
  the analyzer, which then neither keeps nor needs the labelled documents at analysis time — lower
  tokens per analysis, and training data stops travelling with the artefact. It cannot fix
  character-recognition errors: if a letter is misread as a digit, labelling will not help.
- **Copy an analyzer between resources** — `grant_copy_authorization` then `begin_copy_analyzer`,
  for promoting a tested analyzer from dev to production.
- **Wire the output into Azure AI Search** — a built-in skill does extraction and chunking in one
  step.

### Reference

| | |
|---|---|
| Overview and key components | `learn.microsoft.com/azure/ai-services/content-understanding/overview` |
| Analyzer reference | `.../concepts/analyzer-reference` |
| What's new | `.../whats-new` |
| Agentic mode | `.../concepts/agentic-mode` |
| Choosing between tools | `.../choosing-right-ai-tool` |
| Service limits and regions | `.../service-limits` · `.../language-region-support` |
| How billing works | `.../pricing-explainer` |
| Reference architecture | `learn.microsoft.com/azure/architecture/ai-ml/idea/multi-modal-content-processing` |
| Python SDK samples | `github.com/Azure/azure-sdk-for-python` → `sdk/contentunderstanding` |
| Python solution samples | `github.com/Azure-Samples/azure-ai-content-understanding-python` |